**Only want to try the adapter, on a free Colab T4?** Use `podbay/run/podbay_try_free_colab.ipynb` instead: inference only,
llama.cpp with the Q4_K_M base and the adapter GGUF (or transformers 4-bit + PEFT), the 21 PodBayBench tasks base
against adapter, and a task you type. This notebook is the training recipe and needs a larger GPU.

# PodBay RLVR brick: load, compare, (train), evaluate, export

The PodBay recipe on a Colab GPU, as it was run. The brick is a LoRA adapter trained by RLVR with the PodBay guard as
the verifier (`podbay/rlvr/`); it bolts onto the base model without changing the base weights.

Sections: **0 Parameters** · **1 Set up** · **2 Load the base model** · **3 Load the adapter** ·
**4 Brick off vs on on test prompts** · **5 (Optional) Train with grpo_torch.py** · **6 Evaluate** ·
**7 Export the adapter to GGUF** (for `llama-server --lora`, see `podbay/run/README.md`).

Defaults are the brick that was brought back: `podbay/results/rlvr/run16/adapter` (recipe in the next cell; how it
was chosen and what it scores are in `podbay/results/rlvr/EXPERIMENTS.md` and `run16/SUMMARY.md`). Base:
`allenai/Olmo-3-7B-Think` in bf16, thinking block closed. This notebook calls the repository's own modules
(`podbay.rlvr.common`, `.eval`, `.grpo_torch`) and does not copy their logic.

Runtime: GPU with about 20 GB or more (A100 recommended; the bf16 7B base takes about 15 GB), high-RAM. A full
training run (150 updates) took about 30 minutes on an A100 80 GB. The copy `podbay_rlvr_colab.executed.ipynb` next
to this file was run top to bottom on a Colab A100 with training cut to a few steps and a small evaluation.


## 0. Parameters (edit these)

In [ ]:
# ---- code ----
CODE_SOURCE = "git"            # "git": clone REPO_URL@BRANCH; "tarball": unpack CODE_TARBALL; "local": the current directory
REPO_URL    = "https://github.com/grassyhilltop/tiny-cua"   # the repository (must be reachable from Colab)
BRANCH      = "main"
CODE_TARBALL = "/content/podbay_code.tgz"

# ---- base model (podbay/rlvr/common.py) ----
BASE_MODEL    = "allenai/Olmo-3-7B-Think"
BASE_REVISION = "d97e442d7cc678210054dbcc9b440894d62c89a4"
FOUR_BIT      = False           # the brick was trained on the bf16 base; True loads NF4 (needs bitsandbytes)

# ---- the brick ----
# A PEFT adapter directory (adapter_config.json + adapter_model.safetensors), relative to the code root or absolute.
ADAPTER_DIR = "podbay/results/rlvr/run16/adapter"   # sha256 d689377d... (MANIFEST.json); None = base vs base

# ---- inference ----
THINK = "closed"                # "closed" (thinking block closed at once, the training mode) or "budget:256"
TEST_SCENES = ["bench:A1", "bench:B1", "bench:C1", "test:0", "test:5", "test:12"]   # scene ids, as in plug.py

# ---- training (section 5; off by default) ----
TRAIN = False
TRAIN_OUT = "/content/out/run_nb"
# the recipe of run16 (podbay/results/rlvr/run16/MANIFEST.json args): minimal twins, 4 pairs per update, adaptive kind
# weights (floor 0.4), group 6, forced give-up probe on should-refuse screens only, Dave probe, rescue probe,
# LoRA rank 16 alpha 32 on all 32 blocks, lr 1.5e-4, KL 0.05, 150 updates, seed 8, validation every 10 updates
TRAIN_ARGS = ("--steps 150 --batch twins --pairs 4 --adaptive 1 --adaptive-floor 0.4 --group 6 --lr 1.5e-4 "
              "--rank 16 --alpha 32 --layers 32 --kl 0.05 --probe 1 --dave-probe 1 --act-rescue 1 --probe-on refuse "
              "--eval-every 10 --seed 8 --think-per-kind 0")

# ---- evaluation (section 6) ----
EVAL_OUT = "/content/out/eval_nb"
EVAL_ARGS = "--sets test,bench,bench_extra,laws --test-per-kind 30 --laws-n 40 --think closed"

# ---- GGUF export (section 7) ----
EXPORT_GGUF = True
BRICK_NAME  = "podbay-door-olmo3-7b-think"   # -> <BRICK_NAME>.gguf (the committed brick has this name)
LLAMA_CPP_REF = "b12110"        # the llama.cpp release the GGUF evaluation used; pin to your llama-server's version


## 1. Set up

In [ ]:
import os, subprocess, sys
os.environ.setdefault("HF_HOME", "/content/hf")
if CODE_SOURCE == "git":
    if not os.path.exists("/content/tiny-cua"):
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/tiny-cua"], check=True)
    ROOT = "/content/tiny-cua"
elif CODE_SOURCE == "tarball":
    ROOT = "/content/podbay_code"
    os.makedirs(ROOT, exist_ok=True)
    subprocess.run(["tar", "xzf", CODE_TARBALL, "-C", ROOT], check=True)
else:
    ROOT = os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("code at", ROOT)
if ADAPTER_DIR and not os.path.isabs(ADAPTER_DIR):
    ADAPTER_DIR = os.path.join(ROOT, ADAPTER_DIR)
print("brick:", ADAPTER_DIR)


In [ ]:
# The versions the Colab job uses (podbay/rlvr/colab/vm_queue.sh); keep them in step with that file.
!pip -q install "transformers==5.23.0" "peft==0.21.3" "accelerate==1.16.0" "safetensors==0.8.0" "tokenizers==0.23.3" "huggingface-hub==2.4.0" bitsandbytes
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# Generator, reward and guard tests (stdlib, seconds): a broken checkout shows up here, not halfway through training.
!cd {ROOT} && python -m unittest discover -s podbay/guard/tests 2>&1 | tail -2 && python -m unittest discover -s podbay/rlvr/tests -t . 2>&1 | tail -2

## 2. Load the base model

In [ ]:
from podbay.rlvr import common as C, scenarios as S
model, tok = C.load_base(BASE_MODEL, four_bit=FOUR_BIT, revision=BASE_REVISION)
model.eval()
print(type(model).__name__, "layers:", C.n_layers(model))
if torch.cuda.is_available():
    print("GPU memory in use: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

## 3. Load the adapter (the brick)

`C.load_adapter` wraps the base in a PEFT model; `C.brick(model, on)` switches the adapter on or off for a block of
code. The base weights are never changed.

In [ ]:
import json
if ADAPTER_DIR:
    model = C.load_adapter(model, ADAPTER_DIR)
    model.eval()
    cfg = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json")))
    print("brick:", ADAPTER_DIR, "| rank", cfg["r"], "alpha", cfg["lora_alpha"], "| layers", (cfg.get("layers_to_transform") or ["all"])[0], "...")
else:
    print("no ADAPTER_DIR: the 'on' arm below is the same as 'off'")

## 4. Inference on test prompts, brick off and on

In [ ]:
from podbay.rlvr.plug import scene_by_id
for sid in TEST_SCENES:
    sc = scene_by_id(sid)
    print("=" * 100); print(sid, "| kind:", sc["kind"], "| expected:", S.expected(sc)); print(S.render(sc))
    for on in (False, True):
        with C.brick(model, on):
            c = C.complete(model, tok, [sc], think=THINK)[0]
        r = S.reward(sc, c["text"])
        print(f"--- brick {'ON ' if on else 'OFF'}: {c['text'].strip()[:300]}")
        print(f"    reward {r['r']:+} ({r['outcome']})")

In [ ]:
# Free text with the same system prompt (anything you like; the brick's effect on ordinary chat)
PROMPT = "Open the pod bay doors, please."
msgs = [{"role": "system", "content": S.SYSTEM}, {"role": "user", "content": PROMPT}]
s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
if THINK == "closed" and s.endswith("<think>"):
    s += "</think>\n"
ids = tok(s, add_special_tokens=False)["input_ids"]
for on in (False, True):
    with C.brick(model, on):
        g = C.generate(model, tok, [ids], 0.0, 60 if THINK == "closed" else 300)[0]
    print(f"--- brick {'ON ' if on else 'OFF'}: {C.decode(tok, g).strip()}")

## 5. (Optional) Train with `podbay/rlvr/grpo_torch.py`

Runs the trainer as a subprocess (it loads its own copy of the base, so free the notebook's model first). Outputs in
`TRAIN_OUT`: `train.jsonl`, `eval_*.jsonl`, `adapter/` (the selected checkpoint), `MANIFEST.json`. For a run that must
survive the browser closing, use the detached job in `podbay/rlvr/colab/` instead (hard cap, idle release, syncer).

In [ ]:
if TRAIN:
    import gc
    del model; gc.collect(); torch.cuda.empty_cache()
    q = "--four-bit" if FOUR_BIT else ""
    !cd {ROOT} && python -m podbay.rlvr.grpo_torch --model {BASE_MODEL} --revision {BASE_REVISION} {q} --out {TRAIN_OUT} {TRAIN_ARGS} 2>&1 | tail -40
    TRAINED_DIR = os.path.join(TRAIN_OUT, "adapter")
    print("trained brick:", TRAINED_DIR, "(set ADAPTER_DIR to it and re-run sections 2-4 to look at it)")
else:
    print("TRAIN is False: skipped")


## 6. Evaluate

`podbay.rlvr.eval`: brick off and on, per scene kind, on PodBayBench and on held-out laws, with the safeguard table.
Every prompt, completion and reward is saved in `EVAL_OUT` (`eval_<set>_<arm>_<think>.jsonl`, `eval_summary.json`).

In [ ]:
if ADAPTER_DIR:
    try:
        del model
    except NameError:
        pass
    import gc; gc.collect(); torch.cuda.empty_cache()
    q = "" if FOUR_BIT else "--no-4bit"
    !cd {ROOT} && python -m podbay.rlvr.eval --model {BASE_MODEL} --revision {BASE_REVISION} {q} --adapter {ADAPTER_DIR} --out {EVAL_OUT} {EVAL_ARGS} 2>&1 | tail -30
    summ = json.load(open(os.path.join(EVAL_OUT, "eval_summary.json")))
    for k, v in summ.items():
        print(k, v["rates"]["should_act_done"], "act |", v["rates"]["should_refuse_handed_back"], "handed back | Dave", v["rates"]["dave"])
else:
    print("no ADAPTER_DIR: nothing to evaluate")


## 7. Export the adapter to GGUF (bolt onto a GGUF base in llama-server)

`convert_lora_to_gguf.py` needs the adapter and the base model's **config.json only** (no weights; tested in
`podbay/run/README.md`). The result goes in `podbay/bricks/` on the Mac; `PodBay.command` loads it at scale 0 and
the page's brick switch sets scale 1 or 0 via `POST /lora-adapters`.

The brick was trained on the bf16 base and is added to a quantised GGUF base (Q4_K_M or Q8_0) in llama-server;
how far its behaviour carries over was measured through llama-server (`podbay/results/rlvr/run16/gguf/`).


In [ ]:
if EXPORT_GGUF and ADAPTER_DIR:
    if not os.path.exists("/content/llama.cpp"):
        !git clone --depth 1 -b {LLAMA_CPP_REF} https://github.com/ggml-org/llama.cpp /content/llama.cpp
        !pip -q install gguf sentencepiece
    from huggingface_hub import hf_hub_download
    cfg_dir = "/content/base_config_only"
    os.makedirs(cfg_dir, exist_ok=True)
    hf_hub_download(BASE_MODEL, "config.json", revision=BASE_REVISION, local_dir=cfg_dir)
    out = f"/content/out/{BRICK_NAME}.gguf"
    !python /content/llama.cpp/convert_lora_to_gguf.py {ADAPTER_DIR} --base {cfg_dir} --outtype f16 --outfile {out} 2>&1 | tail -3
    !ls -la {out} && sha256sum {out}
    print("Copy", out, "to podbay/bricks/ on the Mac.")
else:
    print("skipped (EXPORT_GGUF False or no ADAPTER_DIR)")